# Phase 1.5 — Fine-tune VietOCR (vgg_seq2seq) trên InkData

Notebook chạy trong **Google Colab GPU** (T4 / L4 / A100 đều OK).

Pipeline 5 bước:
1. Kiểm tra GPU + cài đặt
2. Mount Google Drive và chuẩn bị dataset (`data_line/`)
3. Clone code repo (hoặc upload zip) và tách train/val
4. Fine-tune (~1–2h trên T4 với 20k iters)
5. Eval CER/WER trên test set và copy checkpoint về Drive

> **Trước khi chạy**: tải thư mục `c:\Lily\handw\data\data_line` (giữ nguyên cấu trúc) lên Google Drive tại `MyDrive/handw/data_line/`. Tổng dung lượng ~65 MB.

## 1. GPU + install

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q vietocr==0.3.13 "numpy<2.0" pyyaml

## 2. Mount Drive + khắc sẵn đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/handw')
DATA_ROOT  = DRIVE_ROOT / 'data_line'
CODE_ROOT  = Path('/content/handw')
MODEL_OUT  = DRIVE_ROOT / 'models' / 'vietocr'

assert (DATA_ROOT / 'InkData_line_processed').is_dir(), \
    f'data_line khong tim thay tai {DATA_ROOT}. Hay upload data_line/ len Drive truoc.'
MODEL_OUT.mkdir(parents=True, exist_ok=True)
print('OK:', DATA_ROOT)

## 3. Đưa code vào Colab

**Cách A** (khuyến nghị): nén thư mục repo `handw` thành `handw.zip`, upload vào `MyDrive/handw/handw.zip`, rồi chạy cell dưới.

**Cách B**: nếu đã push repo lên GitHub, bỏ comment dòng `!git clone ...`.

In [ ]:
import zipfile, shutil

if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)

zip_path = DRIVE_ROOT / 'handw.zip'
if zip_path.is_file():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(CODE_ROOT.parent)
    # If the zip root is named 'handw' it lands at /content/handw automatically.
    if not CODE_ROOT.is_dir():
        # Find the extracted folder.
        candidates = [p for p in CODE_ROOT.parent.iterdir() if p.is_dir() and (p / 'src' / 'train').is_dir()]
        assert candidates, 'Khong tim thay folder code sau khi giai nen zip'
        candidates[0].rename(CODE_ROOT)
    print('Unzipped code ->', CODE_ROOT)
else:
    # !git clone https://github.com/<your-user>/handw.git /content/handw
    raise FileNotFoundError(f'Khong tim thay {zip_path}. Hay upload handw.zip len Drive.')

In [ ]:
import sys, os
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print('cwd =', os.getcwd())

## 4. Tách train/val + check vocab

In [ ]:
import yaml
from pathlib import Path

cfg_path = CODE_ROOT / 'configs' / 'train_vietocr.yaml'
with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

# Tro cac duong dan vao Drive
cfg['dataset']['data_root']        = str(DATA_ROOT)
cfg['dataset']['train_annotation'] = str(DATA_ROOT / 'train_split.txt')
cfg['dataset']['valid_annotation'] = str(DATA_ROOT / 'val_split.txt')
cfg['dataset']['test_annotation']  = str(DATA_ROOT / 'test_line_annotation.txt')
cfg['split']['source']             = str(DATA_ROOT / 'train_line_annotation.txt')
cfg['output']['dir']               = str(MODEL_OUT)
cfg['device']                      = 'cuda:0'

patched_cfg = CODE_ROOT / 'configs' / 'train_vietocr_colab.yaml'
with open(patched_cfg, 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
print('patched config ->', patched_cfg)

In [ ]:
!python -m src.train.prepare_data --config configs/train_vietocr_colab.yaml

## 5. Fine-tune (~1–2h trên T4)

Mặc định `iters=20000`, `batch_size=32`. Tăng `iters` cho tập lớn hơn, giảm nếu muốn thử nhanh.

In [ ]:
!python -m src.train.train_vietocr --config configs/train_vietocr_colab.yaml

## 6. Đánh giá CER / WER trên test set

In [ ]:
EXPORT = MODEL_OUT / cfg['output']['export_name']
assert EXPORT.is_file(), f'Khong thay checkpoint tai {EXPORT}'
print('weights ->', EXPORT)

!python -m src.train.evaluate \
    --config configs/train_vietocr_colab.yaml \
    --weights "{EXPORT}" \
    --annotation "{DATA_ROOT / 'test_line_annotation.txt'}" \
    --data-root "{DATA_ROOT}" \
    --save-predictions "{MODEL_OUT / 'test_predictions.json'}"

## 7. (Tuỳ chọn) Dùng checkpoint vừa train trong pipeline OCR

Sau khi xong, tải `vietocr_seq2seq_inkdata.pth` từ Drive về máy local và đặt vào `models/vietocr/`, rồi chạy:

```bash
python -m src.cli --image data/samples/sample.png \
    --rec-model vgg_seq2seq \
    --save-viz
```

Nhớ edit `configs/default.yaml` để set `recognizer.weights_path` trỏ vào file `.pth` vừa copy về.

Test nhanh ngay trên Colab:

In [ ]:
from src.recognizer import TextRecognizer
from PIL import Image
from pathlib import Path
import random

rec = TextRecognizer(model='vgg_seq2seq', device='cuda:0', weights_path=str(EXPORT))

with open(DATA_ROOT / 'test_line_annotation.txt', 'r', encoding='utf-8') as f:
    test_lines = [ln.rstrip('\n').split('\t', 1) for ln in f if '\t' in ln]

random.seed(0)
for img_rel, ref in random.sample(test_lines, 8):
    img = Image.open(DATA_ROOT / img_rel).convert('RGB')
    pred, prob = rec.recognize(img)
    print(f'[{prob:.2f}]  REF: {ref}')
    print(f'        PRED: {pred}')
    print()